In [29]:
import pandas as pd
import plotly.express as px

In [30]:
car_df = pd.read_csv("Data/2015_State_Top10Report_wTotalThefts.csv")

car_df.head()

,index,State,Rank,Make/Model,Model Year,Thefts
0,0,Alabama,1.0,Chevrolet Pickup (Full Size),2005.0,499
1,1,Alabama,2.0,Ford Pickup (Full Size),2006.0,357
2,2,Alabama,3.0,Toyota Camry,2014.0,205
3,3,Alabama,4.0,Nissan Altima,2014.0,191
4,4,Alabama,4.0,Chevrolet Impala,2004.0,191


In [31]:
population = {"Alabama": 4858979, "Alaska": 738432, "Arizona": 6828065, "Arkansas": 2978204, "California": 39144818, 
              "Colorado": 5456574, "Connecticut": 3590886, "Delaware": 945934, "Florida": 20271272, "Georgia": 10214860,
              "Hawaii": 1431603, "Idaho": 1654930, "Illinois": 12859995, "Indiana": 6619680, "Iowa": 3123899,
              "Kansas": 2911505, "Kentucky": 4425092, "Louisiana": 4670724, "Maine": 1329328, "Maryland": 6006401,
              "Massachusetts": 6794422, "Michigan": 9922576, "Minnesota": 5489594, "Mississippi": 2992333, "Missouri": 6083672,
              "Montana": 1032949, "Nebraska": 1896190, "Nevada": 2890845, "New Hampshire": 1330608, "New Jersey": 8958013,
              "New Mexico": 2085109, "New York": 19795791, "North Carolina": 10042802, "North Dakota": 756927, "Ohio": 11613423,
              "Oklahoma": 3911338, "Oregon": 4028977, "Pennsylvania": 12802503, "Rhode Island": 1056298, "South Carolina": 4896146,
              "South Dakota": 858469, "Tennessee": 6600299, "Texas": 27469114, "Utah": 2995919, "Vermont": 626042,
              "Virginia": 8382993, "Washington": 7170351, "West Virginia": 1844128, "Wisconsin": 5771337, "Wyoming": 586107, "District of Columbia" : 672228}

populaion_df = pd.DataFrame(population.items(), columns=["State", "Population"])

populaion_df

,State,Population
0,Alabama,4858979
1,Alaska,738432
2,Arizona,6828065
3,Arkansas,2978204
4,California,39144818
5,Colorado,5456574
6,Connecticut,3590886
7,Delaware,945934
8,Florida,20271272
9,Georgia,10214860


In [32]:
car_df["Thefts"] = pd.to_numeric(
    car_df["Thefts"].astype(str).str.replace(",", ""), 
    errors="coerce"
)

#What State has the most car thefts

In [33]:
thefts_by_state = car_df.groupby("State")["Thefts"].sum().reset_index().sort_values("Thefts", ascending=False)
thefts_by_state

,State,Thefts
4,California,86768.0
43,Texas,25433.0
47,Washington,11008.0
9,Florida,9664.0
10,Georgia,6497.0
5,Colorado,5290.0
2,Arizona,5270.0
13,Illinois,5015.0
22,Michigan,4946.0
37,Oregon,4556.0


#Car Theft by capita

In [57]:
state_code = {'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA', 'Colorado': 'CO', 
              'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 
              'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 
              'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO', 
              'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 
              'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 
              'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 
              'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC'}


thefts_rate = thefts_by_state.merge(populaion_df, on="State", how="left")

thefts_rate["Thefts_per_100k"] = (thefts_rate["Thefts"] / thefts_rate["Population"] * 100_000).round(2)

thefts_rate = thefts_rate.sort_values("Thefts_per_100k", ascending=False)

thefts_rate["State_Code"] = thefts_rate["State"].map(state_code)


thefts_rate.head(10)

,State,Thefts,Population,Thefts_per_100k,State_Code
0,California,86768.0,39144818,221.66,CA
2,Washington,11008.0,7170351,153.52,WA
20,New Mexico,3074.0,2085109,147.43,NM
13,Nevada,3883.0,2890845,134.32,NV
38,District of Columbia,805.0,672228,119.75,DC
9,Oregon,4556.0,4028977,113.08,OR
19,Utah,3103.0,2995919,103.57,UT
5,Colorado,5290.0,5456574,96.95,CO
15,Oklahoma,3757.0,3911338,96.05,OK
1,Texas,25433.0,27469114,92.59,TX


In [65]:
fig = px.choropleth(
    thefts_rate,
    locations="State_Code",
    locationmode="USA-states",
    color="Thefts_per_100k",
    scope="usa",
    color_continuous_scale="Reds",
    hover_name="State",
    hover_data={
        "Thefts_per_100k": ":.2f",
        "Thefts": True,
        "Population": ":,.0f"
    },
    title="Car Theft Rate per 100,000 Residents by State"
)

fig.update_layout(
    title_font_size=18,
    coloraxis_colorbar_title="Thefts per 100k"
)

fig.show()